In [13]:
import pandas as pd
df = pd.read_csv("merged.csv")
print(df.shape)
df.head()

(696, 12)


,Monthly_Increase,Month,Year,Volatility,cpi_pct,fedfunds_rate,unemployment_rate,CIVPART,Monthly_Increase_next_month,cpi_pct_lag1,fedfunds_rate_lag1,unemployment_rate_lag1
0,-3.870003,1,1968,0.068666,3.651861,4.75,3.7,59.2,-3.199997,NaN,NaN,NaN
1,-3.199997,2,1968,0.107240,3.673819,4.75,3.8,59.6,1.089996,3.651861,4.75,3.7
2,1.089996,3,1968,0.132372,4.142164,5.25,3.7,59.6,4.979996,3.673819,4.75,3.8
3,4.979996,4,1968,0.138902,4.155828,6.25,3.5,59.5,0.709999,4.142164,5.25,3.7
4,0.709999,5,1968,0.068541,4.088245,6.13,3.5,59.9,-0.409996,4.155828,6.25,3.5


In [14]:
df.dropna(inplace=True)
df.isna().sum()

Monthly_Increase               0
Month                          0
Year                           0
Volatility                     0
cpi_pct                        0
fedfunds_rate                  0
unemployment_rate              0
CIVPART                        0
Monthly_Increase_next_month    0
cpi_pct_lag1                   0
fedfunds_rate_lag1             0
unemployment_rate_lag1         0
dtype: int64

In [15]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# Use 1-period lags for explanatory variables
df_lag = df.copy()
lag_features = ["cpi_pct", "fedfunds_rate", "unemployment_rate", ]
for col in lag_features:
    df_lag[f"{col}_lag1"] = df_lag[col].shift(1)

# Drop rows created by lagging
df_lag = df_lag.dropna(subset=[f"{col}_lag1" for col in lag_features] + ["Monthly_Increase"])

# Features and target
X = df[["cpi_pct", "unemployment_rate", "fedfunds_rate", "Volatility", "CIVPART"]]
y = df["Monthly_Increase"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model
model = LinearRegression()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Metrics
print("R2:", r2_score(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))

# Coefficients
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})

print(coef_df)
print("Intercept:", model.intercept_)

R2: 0.02290188217276301
MSE: 3889.898983757488
             Feature  Coefficient
0            cpi_pct    -2.645582
1  unemployment_rate     2.961038
2      fedfunds_rate    -0.209047
3         Volatility  -241.859617
4            CIVPART    -2.479553
Intercept: 197.34748699741013
